# Сравнение архитектур эмпатичного диалога

**Датасет:** EmpatheticDialogues (test split, ~2547 диалогов)  
**Задача:** генерация эмпатичного ответа на реплику собеседника  

**Модели:**
| Ключ | Модель | Размер | Провайдер |
|---|---|---|---|
| `mistral-small-3.2` | Mistral-Small-3.2-24B-Instruct-2506 | 24B | local vLLM |
| `qwen3-32b-local` | Qwen3-32B | 32B | local vLLM |
| `llama-3.1-8b-local` | Llama-3.1-8B-Instruct | 8B | local vLLM |

**Архитектуры (9 шт.):**
- `empathy_zero_shot` — прямая генерация без примеров
- `empathy_few_shot` — с 5 фиксированными примерами
- `empathy_ektc` — цепочка: эмоция → знание → рефлексия → ответ
- `empathy_chain` — цепочка: эмоция → причина → стратегия → ответ
- `empathy_debate` — 3 агента-дебатёра + арбитр
- `empathy_loop` — итеративное уточнение ответа
- `empathy_rag` — retrieval-augmented generation (FAISS)
- `empathy_mas_c` — multi-agent selector с двумя генераторами
- `empathy_trace` — трассировка рассуждений

**Метрики:** BLEU-1/2, ROUGE-1/L, BERTScore-F, AvgLen, Accuracy (эмоция), кол-во LLM-вызовов  
**Результаты:** сохраняются в `empathy_multiagent/outputs/<model>_results.csv`

> **Перед запуском:** LLM-сервер надо запустить (`python empathy_multiagent/serve_local.py --model <key>`) и `LOCAL_API_KEY=EMPTY` прописать в `.env`

In [1]:
# !pip install -r empathy_multiagent/requirements.txt

In [10]:
import os, subprocess, json
import pandas as pd
from pathlib import Path


ARCHS = [
    "empathy_zero_shot",
    "empathy_few_shot",
    "empathy_ektc",
    "empathy_chain",
    "empathy_debate",
    "empathy_loop",
    "empathy_rag",
    "empathy_mas_c",
    "empathy_trace",
]

WD  = Path("empathy_multiagent")
OUT = WD / "outputs"

METRIC_MAP = {
    "BLEU-1":            "BLEU-1",
    "BLEU-2":            "BLEU-2",
    "ROUGE-1":           "ROUGE-1",
    "ROUGE-L":           "ROUGE-L",
    "BERTScore-F":       "BERTScore",
    "AvgLen":            "AvgLen",
    "Accuracy (%)":      "Accuracy",
    "Avg calls/example": "Вызовов",
}

_ENV = {**os.environ, "MPLBACKEND": "agg"}

def run_model(model_key):
    for arch in ARCHS:
        subprocess.run(
            ["python3", "run_experiment.py", "--model", model_key, "--arch", arch],
            cwd=WD, check=True, env=_ENV,
        )

def load_table(model_key):
    df = pd.read_csv(OUT / f"{model_key}_results.csv")
    df = df.sort_values("BERTScore", ascending=False).reset_index(drop=True)
    return df

## Модель 1: `mistral-small-3.2` — Mistral Small 3.2 24B

In [11]:
run_model("mistral-small-3.2")

In [12]:
load_table("mistral-small-3.2")

,Архитектура,BLEU-1,BLEU-2,ROUGE-1,ROUGE-L,BERTScore,AvgLen,Accuracy,Вызовов
0,empathy_rag,15.63,5.08,15.17,13.64,0.871261,15.4,54.68,3.0
1,empathy_mas,9.46,2.48,9.04,8.22,0.870928,8.6,56.03,5.0
2,empathy_chain,10.04,2.60,6.20,5.98,0.870115,8.0,53.92,4.0
3,empathy_few_shot,10.54,3.61,9.58,7.09,0.868123,13.1,NaN,1.0
4,empathy_loop,11.30,2.13,8.83,8.01,0.867822,8.9,45.16,7.5
5,empathy_debate,7.21,2.57,5.31,4.93,0.867466,8.1,54.53,5.0
6,empathy_trace,12.57,2.71,9.34,9.15,0.867404,11.0,NaN,4.0
7,empathy_zero_shot,11.23,2.69,11.03,9.17,0.865012,20.5,NaN,1.0
8,empathy_ektc,10.12,3.07,10.70,8.33,0.856142,25.1,NaN,3.7


## Модель 2: `qwen3-32b-local` — Qwen3 32B

In [13]:
run_model("qwen3-32b-local")

In [14]:
load_table("qwen3-32b-local")

,Архитектура,BLEU-1,BLEU-2,ROUGE-1,ROUGE-L,BERTScore,AvgLen,Accuracy,Вызовов
0,empathy_mas,10.51,2.37,11.78,10.22,0.865103,15.0,47.50,5.0
1,empathy_trace,11.41,2.67,10.43,8.83,0.864009,14.1,NaN,4.0
2,empathy_zero_shot,10.73,3.58,11.99,10.04,0.863736,22.2,NaN,1.0
3,empathy_chain,9.44,1.56,9.37,8.54,0.862814,16.3,47.21,4.0
4,empathy_rag,10.17,3.26,11.17,10.59,0.862568,25.8,47.32,3.0
5,empathy_few_shot,10.38,2.40,12.15,9.74,0.861281,20.9,NaN,1.0
6,empathy_loop,9.34,2.07,10.73,8.19,0.859377,18.4,46.82,8.0
7,empathy_debate,10.22,2.12,11.84,9.34,0.859354,18.9,47.51,5.0
8,empathy_ektc,9.50,2.19,11.25,8.54,0.856530,29.8,NaN,3.8


## Модель 3: `llama-3.1-8b-local` — Llama 3.1 8B

In [15]:
run_model("llama-3.1-8b-local")

In [16]:
load_table("llama-3.1-8b-local")

,Архитектура,BLEU-1,BLEU-2,ROUGE-1,ROUGE-L,BERTScore,AvgLen,Accuracy,Вызовов
0,empathy_mas,12.82,3.00,9.13,8.33,0.866511,12.8,43.08,5.0
1,empathy_chain,13.41,2.71,11.19,9.31,0.865672,17.4,42.82,4.0
2,empathy_rag,12.46,4.82,11.47,9.98,0.865213,18.4,43.09,3.0
3,empathy_debate,11.17,2.49,9.01,7.55,0.864009,15.6,42.97,5.0
4,empathy_few_shot,11.95,3.83,11.86,9.67,0.863572,19.9,NaN,1.0
5,empathy_trace,12.13,2.36,11.43,9.60,0.862820,19.2,NaN,4.0
6,empathy_loop,12.11,1.82,9.33,7.14,0.862674,17.1,41.12,8.7
7,empathy_zero_shot,10.79,3.38,13.01,11.01,0.857418,32.1,NaN,1.0
8,empathy_ektc,9.19,3.33,11.87,9.04,0.856326,39.9,NaN,3.7
